## 1. Install Dependencies

In [51]:
import os
os.environ["WANDB_DISABLED"] = "true"

!pip install transformers==4.40.2 accelerate==0.30.1 peft==0.10.0 datasets -q --no-cache-dir

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 2. Imports

In [52]:
import os, gc
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

PyTorch: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


## 3. Load & Preprocess Data

In [53]:
df = pd.read_csv("/kaggle/input/datasets/sahil0716/aitext/Training_Essay_Data.csv")

TEXT_COL  = "text"
LABEL_COL = "generated"

df = df[[TEXT_COL, LABEL_COL]].dropna()
df[TEXT_COL] = df[TEXT_COL].astype(str)

print("Dataset size:", len(df))
print("Label distribution:")
print(df[LABEL_COL].value_counts())
df.head()


Dataset size: 29145
Label distribution:
generated
0    17508
1    11637
Name: count, dtype: int64


,text,generated
0,Car-free cities have become a subject of incre...,1
1,"Car Free Cities Car-free cities, a concept ga...",1
2,A Sustainable Urban Future Car-free cities ...,1
3,Pioneering Sustainable Urban Living In an e...,1
4,The Path to Sustainable Urban Living In an ...,1


## 4. Encode Labels & Split

In [54]:
label_encoder = LabelEncoder()
df["label_encoded"] = label_encoder.fit_transform(df[LABEL_COL])
print("Classes:", label_encoder.classes_)

# Use 20% of data for faster T4 training — remove cap for full training
MAX_SAMPLES = 400_000
df_sampled = df.sample(n=min(MAX_SAMPLES, len(df)), random_state=42)

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df_sampled[TEXT_COL],
    df_sampled["label_encoded"],
    test_size=0.1,
    random_state=42,
    stratify=df_sampled["label_encoded"]
)

print(f"Train: {len(train_texts)} | Test: {len(test_texts)}")

Classes: [0 1]
Train: 26230 | Test: 2915


## 5. Tokenize

In [55]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

MAX_LEN = 128  # T4 has 16GB VRAM — 128 is safe and fast

def tokenize_data(texts, labels):
    ds = Dataset.from_pandas(pd.DataFrame({
        "text":   texts.reset_index(drop=True),
        "labels": labels.reset_index(drop=True)
    }))
    ds = ds.map(
        lambda x: tokenizer(x["text"], truncation=True, padding="max_length", max_length=MAX_LEN),
        batched=True,
        batch_size=1000
    )
    drop_cols = [c for c in ds.column_names if c not in ["input_ids", "attention_mask", "labels"]]
    ds = ds.remove_columns(drop_cols)
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

print("Tokenizing train set...")
train_dataset = tokenize_data(train_texts, train_labels)

print("Tokenizing test set...")
test_dataset = tokenize_data(test_texts, test_labels)

print("Done!", train_dataset)

Tokenizing train set...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/26230 [00:00<?, ? examples/s]

Tokenizing test set...


Map:   0%|          | 0/2915 [00:00<?, ? examples/s]

Done! Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 26230
})


## 6. Load Model

In [56]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model.to("cuda")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total params:     66,955,010
Trainable params: 66,955,010


## 7. Training Configuration (T4 Optimized)

In [57]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/results",

    num_train_epochs=1,

    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,

    # ⚡ Speed + memory optimization
    fp16=True,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # 🚨 FIX: disable evaluation (prevents OOM)
    evaluation_strategy="no",

    # 🚨 FIX: disable checkpoint saving (faster, no disk error)
    save_strategy="no",

    # ✅ Optimizer
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    # Logging
    logging_steps=100,
    logging_dir="/kaggle/working/logs",
    report_to="none",
)

print("Training args configured!")

Training args configured!


## 8. Metrics

In [58]:
def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids

    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc}

## 9. Train

In [59]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=None,   # 🚨 no evaluation (correct)
    tokenizer=tokenizer  # ✅ fixed
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to 

Step,Training Loss
100,0.284900
200,0.065900
300,0.044500
400,0.035600


TrainOutput(global_step=410, training_loss=0.10621932170739988, metrics={'train_runtime': 196.452, 'train_samples_per_second': 133.519, 'train_steps_per_second': 2.087, 'total_flos': 868654966686720.0, 'train_loss': 0.10621932170739988, 'epoch': 1.0})

## 10. Evaluate & Save

In [60]:
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=1)

from sklearn.metrics import classification_report
print(classification_report(
    test_labels.reset_index(drop=True),
    preds
))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


              precision    recall  f1-score   support

           0       1.00      0.96      0.98      1751
           1       0.95      1.00      0.97      1164

    accuracy                           0.98      2915
   macro avg       0.97      0.98      0.97      2915
weighted avg       0.98      0.98      0.98      2915



In [61]:
model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

('./final_model/tokenizer_config.json',
 './final_model/special_tokens_map.json',
 './final_model/vocab.txt',
 './final_model/added_tokens.json',
 './final_model/tokenizer.json')

In [63]:
from huggingface_hub import login
login()

In [66]:
from huggingface_hub import create_repo

create_repo("ai-text-detector", private=False)

RepoUrl('https://huggingface.co/sahil077/ai-text-detector', endpoint='https://huggingface.co', repo_type='model', repo_id='sahil077/ai-text-detector')

In [68]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="./final_model",
    repo_id="sahil077/ai-text-detector",
    repo_type="model"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/sahil077/ai-text-detector/commit/cda72965b7005a32615e74e41000e560235f7d29', commit_message='Upload folder using huggingface_hub', commit_description='', oid='cda72965b7005a32615e74e41000e560235f7d29', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sahil077/ai-text-detector', endpoint='https://huggingface.co', repo_type='model', repo_id='sahil077/ai-text-detector'), pr_revision=None, pr_num=None)